<a href="https://colab.research.google.com/github/geek4s/furniture-object-detector/blob/main/furniture_object_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
project = rf.workspace("furniture-d9qab").project("furniture-hpuyb")
version = project.version(1)
dataset = version.download("yolov11")

loading Roboflow workspace...
loading Roboflow project...


In [ ]:
!pip install -q ultralytics roboflow

In [ ]:
from ultralytics import YOLO
import os

In [ ]:
dataset.location

'/content/Furniture-1'

In [ ]:
model = YOLO("yolo11m.pt")

In [ ]:
from ultralytics.utils.torch_utils import select_device

select_device(0)

Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)


device(type='cuda', index=0)

In [ ]:
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,          # Maximum epochs (training may stop earlier)
    imgsz=640,
    batch=16,
    workers=2,
    patience=10,         # Early stopping
    save=True,           # Save checkpoints
    project="Furniture_Detection",
    name="YOLOv11_Furniture"
)

Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Furniture-1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=YOLOv11_Furniture-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=Tr

In [ ]:
metrics = model.val()

print(metrics)

Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11m summary (fused): 126 layers, 20,050,849 parameters, 0 gradients, 67.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 721.8±218.1 MB/s, size: 27.9 KB)
val: Scanning /content/Furniture-1/valid/labels.cache... 241 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 241/241 50.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 1.7it/s 9.4s
                   all        241        566      0.568      0.432      0.483      0.393
         2-seater-sofa         24         26      0.545      0.769      0.646      0.565
         3-seater-sofa         17         18      0.281      0.444      0.335       0.31
                candle          4          6          1          0          0          0
                carpet          8          8      0.435      0.482      0.473      0.391
          center-table          1        

In [ ]:
metrics = model.val()

print("=" * 50)
print("Model Evaluation")
print("=" * 50)

print(f"Precision       : {metrics.box.mp:.4f}")
print(f"Recall          : {metrics.box.mr:.4f}")
print(f"mAP@50          : {metrics.box.map50:.4f}")
print(f"mAP@50-95       : {metrics.box.map:.4f}")

Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1590.8±413.4 MB/s, size: 35.2 KB)
val: Scanning /content/Furniture-1/valid/labels.cache... 241 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 241/241 101.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 1.6it/s 10.2s
                   all        241        566      0.568      0.432      0.483      0.393
         2-seater-sofa         24         26      0.545      0.769      0.646      0.565
         3-seater-sofa         17         18      0.281      0.444      0.335       0.31
                candle          4          6          1          0          0          0
                carpet          8          8      0.435      0.482      0.473      0.391
          center-table          1          1          0          0      0.142      0.114
                 chair          4 

In [ ]:
import os

for root, dirs, files in os.walk("runs"):
    for file in files:
        if file == "best.pt":
            print(os.path.join(root, file))

runs/detect/Furniture_Detection/YOLOv11_Furniture/weights/best.pt
runs/detect/Furniture_Detection/YOLOv11_Furniture-2/weights/best.pt


In [ ]:
best_model = YOLO("runs/detect/Furniture_Detection/YOLOv11_Furniture-2/weights/best.pt")

In [ ]:
test_metrics = best_model.val(
    data=f"{dataset.location}/data.yaml",
    split="val" # Changed from 'valid' to 'val' to match common Ultralytics data.yaml conventions
)

Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1708.6±823.0 MB/s, size: 53.7 KB)
val: Scanning /content/Furniture-1/valid/labels.cache... 241 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 241/241 91.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 1.8it/s 9.1s
                   all        241        566      0.568      0.432      0.483      0.393
         2-seater-sofa         24         26      0.545      0.769      0.646      0.565
         3-seater-sofa         17         18      0.281      0.444      0.335       0.31
                candle          4          6          1          0          0          0
                carpet          8          8      0.435      0.482      0.473      0.391
          center-table          1          1          0          0      0.142      0.114
                 chair          4   

In [ ]:
print("=" * 50)
print("Furniture Detection - Test Set Results")
print("=" * 50)

print(f"Precision : {test_metrics.box.mp*100:.2f}%")
print(f"Recall    : {test_metrics.box.mr*100:.2f}%")
print(f"mAP@50    : {test_metrics.box.map50*100:.2f}%")
print(f"mAP50-95  : {test_metrics.box.map*100:.2f}%")

Furniture Detection - Test Set Results
Precision : 56.84%
Recall    : 43.18%
mAP@50    : 48.31%
mAP50-95  : 39.35%
